In [81]:
%matplotlib inline

In [58]:
#Import your libraries here

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import itertools
import os
import sys
from pathlib import Path
from scipy import stats as st
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [3]:
notebook_path = Path(os.getcwd()).resolve()

def get_root(path):
    for parent in [path] + list(path.parents):
        if (parent / "static_data").exists():
            return parent
    return path

PROJECT_ROOT = get_root(notebook_path)
DATA_DIR = PROJECT_ROOT / "static_data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Source:  {DATA_DIR}")

Project Root: D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project
Data Source:  D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project\static_data


In [80]:
import importlib

BASE_URL = "https://storage.googleapis.com/softuni_data_science_final_project_kamend1/static_data/"

from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.plotting_utils.plotting_utils as plot_utils
import signal_testing_utils.signal_testing_utils as sig_utils
importlib.reload(plot_utils)
importlib.reload(data_pipe)
importlib.reload(sig_utils)

ModuleNotFoundError: No module named 'signal_testing_utils'

In [8]:
tickers = [
    "AAPL", "GOOG", "MSFT", "NVDA",
    "JPM", "BAC", "F", "UPS", "WMT", "TGT",
    "VZ", "T", "FE", "PFE", "JNJ", "DIS",
    "V", "MCD", "NKE", "XOM", "CVX",
    "CAT", "DE", "LMT", "AMD", "INTC", "ORCL", 
    "CRM", "CB", "PG"
]

In [15]:
returns_df =  pd.read_csv(BASE_URL + "stock_returns_static_dataset.csv")
returns_df['Date'] = pd.to_datetime(returns_df['Date'])
returns_df_long = returns_df.melt(id_vars=['Date'], var_name='Ticker', value_name='log_return')
returns_df_long

,Date,Ticker,log_return
0,2016-04-01,AAPL,0.009133
1,2016-04-04,AAPL,0.010221
2,2016-04-05,AAPL,-0.011859
3,2016-04-06,AAPL,0.010418
4,2016-04-07,AAPL,-0.022051
...,...,...,...
75385,2026-03-24,PG,-0.005781
75386,2026-03-25,PG,0.005295
75387,2026-03-26,PG,-0.010477
75388,2026-03-27,PG,0.002034


In [17]:
close_price_df =  pd.read_csv(BASE_URL + "stock_close_prices_static_dataset.csv")
close_price_df['Date'] = pd.to_datetime(returns_df['Date'])
close_price_df_long = close_price_df.melt(id_vars=['Date'], var_name='Ticker', value_name='close_price')
close_price_df_long

,Date,Ticker,close_price
0,2016-04-01,AAPL,24.684103
1,2016-04-04,AAPL,24.910585
2,2016-04-05,AAPL,25.166506
3,2016-04-06,AAPL,24.869822
4,2016-04-07,AAPL,25.130268
...,...,...,...
75415,2026-03-25,PG,143.160004
75416,2026-03-26,PG,143.919998
75417,2026-03-27,PG,142.419998
75418,2026-03-30,PG,142.710007


In [35]:
price_return_df_long = returns_df_long.merge(close_price_df_long, how='left', on=['Date', 'Ticker'])
price_return_df_long = price_return_df_long.sort_values(['Ticker', 'Date'])
price_return_df_long

,Date,Ticker,log_return,close_price
0,2016-04-01,AAPL,0.009133,24.684103
1,2016-04-04,AAPL,0.010221,24.910585
2,2016-04-05,AAPL,-0.011859,25.166506
3,2016-04-06,AAPL,0.010418,24.869822
4,2016-04-07,AAPL,-0.022051,25.130268
...,...,...,...,...
50255,2026-03-24,XOM,0.026034,161.130005
50256,2026-03-25,XOM,-0.012902,165.380005
50257,2026-03-26,XOM,0.013204,163.259995
50258,2026-03-27,XOM,0.033057,165.429993


In [76]:
# def calculate_rsi(series, window=14):
#     """
#     Calculates the Raw RSI value (0-100).
#     Does not generate signals yet.
#     """
#     delta = series.diff()

#     gain = delta.clip(lower=0)
#     loss = -1 * delta.clip(upper=0)

#     avg_gain = gain.rolling(window=window, min_periods=window).mean()
#     avg_loss = loss.rolling(window=window, min_periods=window).mean()
#     rs = avg_gain / avg_loss.replace(0, np.nan)

#     rsi = 100 - (100 / (1 + rs))

#     return rsi


# def calculate_macd_all(series, fast=12, slow=26, signal=9):
#     """
#     Calculates and returns all MACD components as a DataFrame.
#     """
#     ema_fast = series.ewm(span=fast, adjust=False).mean()
#     ema_slow = series.ewm(span=slow, adjust=False).mean()

#     macd_line = ema_fast - ema_slow
#     signal_line = macd_line.ewm(span=signal, adjust=False).mean()
#     macd_hist = macd_line - signal_line

#     return pd.DataFrame({
#         'macd_line': macd_line,
#         'macd_signal': signal_line,
#         'macd_hist': macd_hist
#     }, index=series.index)


# def calculate_volatility(series, window=20):
#     """
#     Calculates the rolling standard deviation of log returns.
#     Represents the 'Risk' or 'Uncertainty' pillar.
#     """
#     return series.rolling(window=window).std()


# def calculate_ma_dist(series, window=50):
#     """
#     Calculates the percentage distance from the Simple Moving Average.
#     Formula: (Price - SMA) / SMA
#     Represents the 'Mean Reversion' or 'Value' pillar.
#     """
#     sma = series.rolling(window=window).mean()
#     return (series - sma) / sma


# def technical_indicator_engine(df):
#     """
#     Computes all 4 technical pillars and cleans up formatting/NaNs.
#     """
#     df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

#     df['rsi_14'] = df.groupby('Ticker')['close_price'].transform(calculate_rsi)

#     macd_cols = (
#         df.groupby('Ticker')['close_price']
#         .apply(calculate_macd_all)
#         .reset_index(level=0, drop=True)
#     )
#     df = pd.concat([df, macd_cols], axis=1)

#     df['volatility_20'] = df.groupby('Ticker')['log_return'].transform(calculate_volatility)

#     df['ma_50_dist'] = df.groupby('Ticker')['close_price'].transform(calculate_ma_dist)

#     technical_cols = ['rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'volatility_20', 'ma_50_dist']
#     df[technical_cols] = df[technical_cols].fillna(0)

#     df['target_log_return_t1'] = df.groupby('Ticker')['log_return'].shift(-1)
#     df = df.dropna(subset=['target_log_return_t1']).copy()

#     return df


# def run_individual_hypothesis_tests(df, target):
#     indicators = ['rsi_14', 'macd_hist', 'volatility_20', 'ma_50_dist']
#     target = target

#     print(f"{'Indicator':<15} | {'Correlation':<12} | {'P-Value':<10} | {'Significant?'}")
#     print("-" * 60)

#     for ind in indicators:
#         # Calculate Pearson correlation and two-tailed p-value
#         corr, p_value = st.pearsonr(df[ind], df[target])

#         is_sig = "YES" if p_value < 0.01 else "NO"
#         print(f"{ind:<15} | {corr:>12.6f} | {p_value:>10.4e} | {is_sig}")


# def run_multivariate_test(df, target):
#     features = ['rsi_14', 'macd_hist', 'volatility_20', 'ma_50_dist']
#     X = df[features].copy()
#     y = df[target]
    
#     # Environment Interactions (The ones we ran)
#     X['RSI_x_Vol'] = X['rsi_14'] * X['volatility_20']
#     X['MACD_x_Vol'] = X['macd_hist'] * X['volatility_20']
    
#     # Internal Technical Interactions (The missing ones)
#     X['RSI_x_MACD'] = X['rsi_14'] * X['macd_hist']
#     X['RSI_x_MA'] = X['rsi_14'] * X['ma_50_dist']
#     X['Vol_x_MA'] = X['volatility_20'] * X['ma_50_dist']
    
#     X = sm.add_constant(X)
#     model = sm.OLS(y, X).fit()
#     return model


# def create_categorical_signals(df):
#     """
#     Converts raw technical indicators into categorical signals (-1, 0, 1).
#     Ensures zero data leakage between tickers via groupby.
#     """
#     # 1. RSI Fixed Thresholds
#     # Capture extreme momentum (Oversold < 30, Overbought > 70)
#     df['sig_rsi'] = np.select(
#         [df['rsi_14'] < 30, df['rsi_14'] > 70], 
#         [1, -1], default=0
#     )
    
#     # 2. MACD Crossovers 
#     # Capture trend inflection points (Zero-line crossing)
#     hist = df['macd_hist']
#     prev_hist = df.groupby('Ticker')['macd_hist'].shift(1)
    
#     df['sig_macd'] = np.select(
#         [(hist > 0) & (prev_hist <= 0), (hist < 0) & (prev_hist >= 0)],
#         [1, -1], default=0
#     )
    
#     # 3. Volatility Quantiles
#     # Capture relative risk (Stability vs. Panic)
#     def get_vol_quantiles(x):
#         q25, q75 = x.quantile(0.25), x.quantile(0.75)
#         return np.select([x < q25, x > q75], [1, -1], default=0)
    
#     df['sig_vol'] = df.groupby('Ticker')['volatility_20'].transform(get_vol_quantiles)
    
#     # 4. MA Distance (Bollinger/Mean Reversion Logic)
#     # Capture statistical outliers relative to the 50-day average
#     def get_ma_zscore_signal(x):
#         sma = x.rolling(window=50).mean()
#         std = x.rolling(window=50).std()
#         z_score = (x - sma) / std
#         # Buy if price is > 2 StdDevs below SMA, Sell if > 2 StdDevs above
#         return np.select([z_score < -2, z_score > 2], [1, -1], default=0)
    
#     df['sig_ma_dist'] = df.groupby('Ticker')['close_price'].transform(get_ma_zscore_signal)
    
#     # Fill any new NaNs from the 50-day rolling window with 0 (Neutral)
#     signal_cols = ['sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist']
#     df[signal_cols] = df[signal_cols].fillna(0).astype(int)
    
#     return df


# def verify_signal_performance(df):
#     signals = ['sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist']
#     target = 'target_log_return_t1'
    
#     performance_metrics = []
    
#     for sig in signals:
#         # Calculate mean returns for each category
#         stats_by_sig = df.groupby(sig)[target].agg(['mean', 'std', 'count'])
        
#         # Perform T-Test: Compare Buy (+1) returns vs Neutral (0) returns
#         buy_returns = df[df[sig] == 1][target]
#         neutral_returns = df[df[sig] == 0][target]
#         t_stat, p_val = st.ttest_ind(buy_returns, neutral_returns, nan_policy='omit')
        
#         performance_metrics.append({
#             'Signal': sig,
#             'Buy_Mean_Return': stats_by_sig.loc[1, 'mean'] if 1 in stats_by_sig.index else np.nan,
#             'Neutral_Mean_Return': stats_by_sig.loc[0, 'mean'],
#             'T_Statistic': t_stat,
#             'P_Value': p_val,
#             'Significant_01': p_val < 0.01
#         })
        
#     return pd.DataFrame(performance_metrics)


# def run_multivariate_anova(df):
#     signals = ['sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist']
#     target = 'target_log_return_t1'
    
#     anova_results = []
#     for sig in signals:
#         # Group returns by the three categories: -1, 0, 1
#         group_neg = df[df[sig] == -1][target]
#         group_neu = df[df[sig] == 0][target]
#         group_pos = df[df[sig] == 1][target]
        
#         # Run One-Way ANOVA
#         f_stat, p_val = st.f_oneway(group_neg, group_neu, group_pos)
        
#         anova_results.append({
#             'Indicator': sig,
#             'F_Statistic': f_stat,
#             'P_Value': p_val,
#             'Reject_H0_01': p_val < 0.01
#         })
#     return pd.DataFrame(anova_results)


# def run_signal_chi2(df):
#     # Create a binary Win/Loss outcome
#     df['outcome_win'] = (df['target_log_return_t1'] > 0).astype(int)
    
#     indicators = ['sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist']
#     chi2_results = []
    
#     for sig in indicators:
#         # Create a Contingency Table (Cross-tabulation)
#         contingency_table = pd.crosstab(df[sig], df['outcome_win'])
        
#         chi2, p, dof, ex = st.chi2_contingency(contingency_table)
        
#         chi2_results.append({
#             'Indicator': sig,
#             'Chi2_Stat': chi2,
#             'P_Value': p,
#             'Significant_01': p < 0.01
#         })
#     return pd.DataFrame(chi2_results)


# def run_categorical_chi2(df):
#     # 1. Define 'Win' as any positive return
#     df['outcome_win'] = (df['target_log_return_t1'] > 0).astype(int)
    
#     indicators = ['sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist']
#     chi2_results = []
    
#     for sig in indicators:
#         # Create a Contingency Table: Rows (Signal: -1, 0, 1) x Columns (Win: 0, 1)
#         contingency = pd.crosstab(df[sig], df['outcome_win'])
        
#         # Run Chi-Squared Test of Independence
#         chi2, p, dof, ex = st.chi2_contingency(contingency)
        
#         chi2_results.append({
#             'Indicator': sig,
#             'Chi2_Stat': chi2,
#             'P_Value': p,
#             'Significant_01': p < 0.01
#         })
#     return pd.DataFrame(chi2_results)


# def add_multi_day_targets(df, horizons=[2, 3, 5]):
#     """
#     Creates cumulative forward log returns for different time horizons.
#     """
#     for n in horizons:
#         col_name = f'target_log_return_t{n}'
#         # Summing the next N days of log returns per ticker
#         # We shift -1, -2, ... -n and sum them
#         df[col_name] = df.groupby('Ticker')['log_return'].transform(
#             lambda x: sum(x.shift(-i) for i in range(1, n + 1))
#         )
    
#     # Drop rows where the longest horizon (t5) is NaN
#     return df.dropna(subset=['target_log_return_t5']).copy()

# def normalize_targets_per_ticker(df, target_cols):
#     """
#     Normalizes (Z-scores) the target returns within each ticker group.
#     This aligns the targets with the sentiment logic.
#     """
#     for col in target_cols:
#         df[f'norm_{col}'] = df.groupby('Ticker')[col].transform(
#             lambda x: (x - x.mean()) / x.std()
#         )
#     return df

In [36]:
price_return_df_long = technical_indicator_engine(price_return_df_long)
price_return_df_long

,Date,Ticker,log_return,close_price,rsi_14,macd_line,macd_signal,macd_hist,volatility_20,ma_50_dist,target_log_return_t1
0,2016-04-01,AAPL,0.009133,24.684103,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010221
1,2016-04-04,AAPL,0.010221,24.910585,0.000000,0.018067,0.003613,0.014454,0.000000,0.000000,-0.011859
2,2016-04-05,AAPL,-0.011859,25.166506,0.000000,0.052431,0.013377,0.039054,0.000000,0.000000,0.010418
3,2016-04-06,AAPL,0.010418,24.869822,0.000000,0.055090,0.021720,0.033371,0.000000,0.000000,-0.022051
4,2016-04-07,AAPL,-0.022051,25.130268,0.000000,0.077322,0.032840,0.044482,0.000000,0.000000,0.001105
...,...,...,...,...,...,...,...,...,...,...,...
75384,2026-03-23,XOM,0.009102,159.669998,61.925600,3.868865,3.650743,0.218122,0.012172,0.104942,0.026034
75385,2026-03-24,XOM,0.026034,161.130005,71.213512,4.034743,3.727543,0.307200,0.012749,0.109054,-0.012902
75386,2026-03-25,XOM,-0.012902,165.380005,82.201977,4.457756,3.873585,0.584170,0.013307,0.131826,0.013204
75387,2026-03-26,XOM,0.013204,163.259995,74.664536,4.569257,4.012720,0.556538,0.013300,0.111225,0.033057


In [68]:
target = "target_log_return_t1"
run_individual_hypothesis_tests(price_return_df_long, target)

Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.008726 | 1.6603e-02 | NO
macd_hist       |    -0.002684 | 4.6127e-01 | NO
volatility_20   |     0.016624 | 5.0257e-06 | YES
ma_50_dist      |    -0.006316 | 8.2938e-02 | NO


In [69]:
target = "target_log_return_t1"
multivariate_model = run_multivariate_test(price_return_df_long, target)
multivariate_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     target_log_return_t1   R-squared:                       0.000
Model:                              OLS   Adj. R-squared:                  0.000
Method:                   Least Squares   F-statistic:                     3.897
Date:                  Tue, 07 Apr 2026   Prob (F-statistic):           5.80e-05
Time:                          15:15:48   Log-Likelihood:             1.9094e+05
No. Observations:                 75360   AIC:                        -3.819e+05
Df Residuals:                     75350   BIC:                        -3.818e+05
Df Model:                             9                                         
Covariance Type:              nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0006      0.001      1.204      0.229      -0.000       0.002
rsi_14        -1.086e-05   9.63e-06     -1.128      0.259   -2.97e-05    8.02e-06
macd_hist         0.0003      0.000      1.092      0.275      -0.000       0.001
volatility_20     0.0516      0.031      1.648      0.099      -0.010       0.113
ma_50_dist        0.0087      0.004      2.105      0.035       0.001       0.017
RSI_x_Vol        -0.0004      0.001     -0.621      0.535      -0.002       0.001
MACD_x_Vol        0.0019      0.006      0.294      0.769      -0.011       0.014
RSI_x_MACD    -5.257e-06   4.18e-06     -1.258      0.208   -1.34e-05    2.93e-06
RSI_x_MA      -6.606e-05   5.69e-05     -1.160      0.246      -0.000    4.55e-05
Vol_x_MA         -0.1201      0.073     -1.654      0.098      -0.262       0.022
==============================================================================
Omnibus:                    18070.327   Durbin-Watson:                   2.110
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1114602.904
Skew:                          -0.135   Prob(JB):                         0.00
Kurtosis:                      21.839   Cond. No.                     6.42e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.42e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Ok, this officially is a complete nightmare

None of these seem to work. Neither Separately, nor together

### So Let's Move to Creating signals and testing them

We use three distinct mathematical methods to create these categories, depending on the nature of the indicator.1. Fixed Thresholds (The "Extreme" Method)Used for indicators with a bounded scale, like RSI (0-100). We ignore the "Middle Noise" (40–60) and only categorize the extremes.Logic:-1 (Sell): $RSI > 70$ (Overbought)0 (Neutral): $RSI$ between $30$ and $70$+1 (Buy): $RSI < 30$ (Oversold)Why: This handles the Non-Linearity your regression struggled with. It tells the model: "Only pay attention when the rubber band is stretched to the limit."2. Crossover Logic (The "Momentum Flip" Method)Used for oscillating indicators like MACD Histogram. The absolute value (0.002 vs 0.003) matters less than the Direction of the Trend.Logic:+1 (Buy): Today's Histogram $> 0$ AND Yesterday's $\leq 0$ (Bullish Cross)-1 (Sell): Today's Histogram $< 0$ AND Yesterday's $\geq 0$ (Bearish Cross)0 (Neutral): All other states.Why: This captures inflection points. It turns a continuous stream of decimals into a binary "Go/No-Go" decision.3. Statistical Quantiles (The "Relative" Method)Used for unbounded numbers like Volatility or MA Distance. Since "High Volatility" for a tech stock like NVDA is "Low Volatility" for a utility stock like Duke Energy, we cannot use fixed numbers. We use the stock's own history.Logic: * Calculate the 25th and 75th percentiles of the indicator for that specific ticker.+1 (Buy): Current value is in the Bottom 25% (Historical Stability/Deep Discount).-1 (Sell): Current value is in the Top 25% (Historical Panic/Extreme Extension).0 (Neutral): The "Normal" 50% in the middle.Why: This Normalizes the data. It ensures that a signal in XOM is mathematically comparable to a signal in AAPL, despite their different price scales.

In [48]:
final_signal_df = create_categorical_signals(price_return_df_long)
final_signal_df

,Date,Ticker,log_return,close_price,rsi_14,macd_line,macd_signal,macd_hist,volatility_20,ma_50_dist,target_log_return_t1,sig_rsi,sig_macd,sig_vol,sig_ma_dist
0,2016-04-01,AAPL,0.009133,24.684103,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010221,1,0,1,0
1,2016-04-04,AAPL,0.010221,24.910585,0.000000,0.018067,0.003613,0.014454,0.000000,0.000000,-0.011859,1,1,1,0
2,2016-04-05,AAPL,-0.011859,25.166506,0.000000,0.052431,0.013377,0.039054,0.000000,0.000000,0.010418,1,0,1,0
3,2016-04-06,AAPL,0.010418,24.869822,0.000000,0.055090,0.021720,0.033371,0.000000,0.000000,-0.022051,1,0,1,0
4,2016-04-07,AAPL,-0.022051,25.130268,0.000000,0.077322,0.032840,0.044482,0.000000,0.000000,0.001105,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75384,2026-03-23,XOM,0.009102,159.669998,61.925600,3.868865,3.650743,0.218122,0.012172,0.104942,0.026034,0,0,0,0
75385,2026-03-24,XOM,0.026034,161.130005,71.213512,4.034743,3.727543,0.307200,0.012749,0.109054,-0.012902,-1,0,0,0
75386,2026-03-25,XOM,-0.012902,165.380005,82.201977,4.457756,3.873585,0.584170,0.013307,0.131826,0.013204,-1,0,0,0
75387,2026-03-26,XOM,0.013204,163.259995,74.664536,4.569257,4.012720,0.556538,0.013300,0.111225,0.033057,-1,0,0,0


In [52]:
signal_alpha = verify_signal_performance(final_signal_df)
signal_alpha

,Signal,Buy_Mean_Return,Neutral_Mean_Return,T_Statistic,P_Value,Significant_01
0,sig_rsi,0.000607,0.000606,0.002122,0.998307,False
1,sig_macd,0.001089,0.000521,1.587151,0.112483,False
2,sig_vol,0.000422,0.000454,-0.216677,0.828461,False
3,sig_ma_dist,0.000561,0.000570,-0.025859,0.979370,False


In [54]:
anova_df = run_multivariate_anova(final_signal_df)
anova_df

,Indicator,F_Statistic,P_Value,Reject_H0_01
0,sig_rsi,1.799542,0.165382,False
1,sig_macd,1.265993,0.281965,False
2,sig_vol,3.136641,0.043434,False
3,sig_ma_dist,0.664606,0.514479,False


In [56]:
chi2_df = run_signal_chi2(final_signal_df)
chi2_df

,Indicator,Chi2_Stat,P_Value,Significant_01
0,sig_rsi,3.030120,0.219795,False
1,sig_macd,2.385341,0.303410,False
2,sig_vol,0.529651,0.767340,False
3,sig_ma_dist,2.441295,0.295039,False


In [59]:
formula = 'target_log_return_t1 ~ C(sig_rsi) + C(sig_vol) + C(sig_rsi):C(sig_vol)'
model = ols(formula, data=final_signal_df).fit()
aov_table = sm.stats.anova_lm(model, typ=2)

print("\n--- Interaction ANOVA (RSI x Volatility) ---")
print(aov_table)


--- Interaction ANOVA (RSI x Volatility) ---
                          sum_sq       df         F    PR(>F)
C(sig_rsi)              0.001155      2.0  1.565571  0.208976
C(sig_vol)              0.002141      2.0  2.902832  0.054874
C(sig_rsi):C(sig_vol)   0.005774      4.0  3.913722  0.003521
Residual               27.792171  75351.0       NaN       NaN


In [62]:
chi2_summary = run_categorical_chi2(final_signal_df)
print(chi2_summary)

     Indicator  Chi2_Stat   P_Value  Significant_01
0      sig_rsi   3.030120  0.219795           False
1     sig_macd   2.385341  0.303410           False
2      sig_vol   0.529651  0.767340           False
3  sig_ma_dist   2.441295  0.295039           False


In [64]:
final_signal_df['confluence_score'] = (
    final_signal_df['sig_rsi'] + 
    final_signal_df['sig_macd'] + 
    final_signal_df['sig_ma_dist']
)

# Test if High Confluence (score >= 2) has a better Win-Rate than neutral (0)
contingency_conf = pd.crosstab(final_signal_df['confluence_score'], final_signal_df['outcome_win'])
chi2_c, p_c, dof_c, ex_c = st.chi2_contingency(contingency_conf)

print(f"\n--- Confluence Interaction (Chi-Squared) ---")
print(f"P-Value for Confluence Score: {p_c:.4e}")
print(f"Significant at 0.01? {'YES' if p_c < 0.01 else 'NO'}")


--- Confluence Interaction (Chi-Squared) ---
P-Value for Confluence Score: 4.8311e-01
Significant at 0.01? NO


These results provide the first piece of objective "Alpha" for your project. While the individual indicators failed, the Interaction ANOVA finally broke the $0.01$ significance threshold.Here is the objective breakdown:1. The Interaction Success: C(sig_rsi):C(sig_vol)P-Value: $0.003521$Verdict: Reject the Null Hypothesis ($H_0$) for Interaction.Significance: This is your most important finding. It proves that RSI does not act alone. Its influence on the magnitude of tomorrow's return is statistically dependent on the Volatility regime.The "Why": A "Buy" signal in a low-volatility environment behaves differently than a "Buy" signal in a high-volatility one. The interaction term captures this conditional logic where the individual terms missed it.2. The Failure of "Confluence" (Chi-Squared)P-Value: $0.4831$Verdict: Fail to Reject the Null Hypothesis.Insight: Simply adding indicators together (sig_rsi + sig_macd + sig_ma_dist) does not improve your "Win Probability."Critical Interpretation: This suggests that technical indicators are not "cumulative." Having three indicators agree doesn't make a win more likely than having one. Interaction is about how they modulate each other (like the RSI x Volatility result), not just stacking them up.3. ANOVA vs. Chi-Squared ContrastANOVA (Magnitude): Found significance in the interaction ($P = 0.003$).Chi-Squared (Frequency): Found nothing ($P > 0.20$ for all).Objective Conclusion: Your indicators affect the size of the next move (Volatility/Magnitude) but they do not reliably predict the direction (Win/Loss frequency). In quantitative terms: you have found an edge in "Expected Value" (due to magnitude) rather than "Hit Rate."Revised Hypothesis StatusHypothesis SetStatusObjective Reality1-4: Individual AlphaFAILEDNone of the 4 indicators work in isolation.5: Multivariate (All)WEAKThe "Team" is better than the individual, but only slightly.6: InteractionSUCCESSRSI x Volatility is a statistically significant relationship.What you are missing for the final defense:The data shows that Volatility is the "Master Variable." It is the only thing that showed up in the raw correlation, the individual ANOVA ($P=0.054$, near-significant), and the Interaction term ($P=0.003$).To conclude the statistical verification phase, we need to answer: "In which direction does the RSI x Volatility interaction work?" (e.g., Does RSI work better when Volatility is high or low?).

In [75]:
price_return_multiday_df = add_multi_day_targets(final_signal_df)

cols_to_drop = [
    'sig_rsi', 'sig_macd', 'sig_vol', 'sig_ma_dist', 
    'outcome_win', 'confluence_score'
]

price_return_multiday_df = price_return_multiday_df.drop(columns=cols_to_drop, errors='ignore')

price_return_multiday_df

,Date,Ticker,log_return,close_price,rsi_14,macd_line,macd_signal,macd_hist,volatility_20,ma_50_dist,target_log_return_t1,target_log_return_t2,target_log_return_t3,target_log_return_t5
0,2016-04-01,AAPL,0.009133,24.684103,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010221,-0.001638,0.008780,-0.012166
1,2016-04-04,AAPL,0.010221,24.910585,0.000000,0.018067,0.003613,0.014454,0.000000,0.000000,-0.011859,-0.001441,-0.023492,-0.019079
2,2016-04-05,AAPL,-0.011859,25.166506,0.000000,0.052431,0.013377,0.039054,0.000000,0.000000,0.010418,-0.011633,-0.010528,0.005721
3,2016-04-06,AAPL,0.010418,24.869822,0.000000,0.055090,0.021720,0.033371,0.000000,0.000000,-0.022051,-0.020946,-0.017638,0.009686
4,2016-04-07,AAPL,-0.022051,25.130268,0.000000,0.077322,0.032840,0.044482,0.000000,0.000000,0.001105,0.004413,0.017354,0.032273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75379,2026-03-16,XOM,0.007085,156.119995,60.823913,3.157060,3.526116,-0.369056,0.015956,0.109174,0.009999,0.002287,0.005898,0.024502
75380,2026-03-17,XOM,0.009999,157.229996,66.352078,3.374751,3.495843,-0.121092,0.015431,0.111110,-0.007712,-0.004101,0.005401,0.040537
75381,2026-03-18,XOM,-0.007712,158.809998,68.932039,3.632889,3.523252,0.109637,0.014347,0.116441,0.003611,0.013112,0.022215,0.035347
75382,2026-03-19,XOM,0.003611,157.589996,67.107758,3.696411,3.557884,0.138527,0.014350,0.102739,0.009502,0.018604,0.044639,0.044941


In [71]:
targets = ["target_log_return_t2", "target_log_return_t3", "target_log_return_t5"]

for target in targets:
    run_individual_hypothesis_tests(price_return_multiday_df, target)

Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.013333 | 2.5566e-04 | YES
macd_hist       |    -0.004103 | 2.6046e-01 | NO
volatility_20   |     0.026546 | 3.3080e-13 | YES
ma_50_dist      |    -0.014846 | 4.6683e-05 | YES
Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.016391 | 6.9448e-06 | YES
macd_hist       |    -0.003010 | 4.0909e-01 | NO
volatility_20   |     0.033919 | 1.3461e-20 | YES
ma_50_dist      |    -0.018338 | 4.9210e-07 | YES
Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.014360 | 8.2072e-05 | YES
macd_hist       |     0.000609 | 8.6735e-01 | NO
volatility_20   |     0.047537 | 6.8869e-39 | YES
ma_50_dist      |    -0.020126 | 3.3915e-08 | YES


In [74]:
targets = ["target_log_return_t2", "target_log_return_t3", "target_log_return_t5"]

for target in targets:
    multivariate_model = run_multivariate_test(price_return_multiday_df, target)
    print(multivariate_model.summary())

                             OLS Regression Results                             
Dep. Variable:     target_log_return_t2   R-squared:                       0.001
Model:                              OLS   Adj. R-squared:                  0.001
Method:                   Least Squares   F-statistic:                     11.23
Date:                  Tue, 07 Apr 2026   Prob (F-statistic):           9.69e-18
Time:                          15:19:14   Log-Likelihood:             1.6666e+05
No. Observations:                 75210   AIC:                        -3.333e+05
Df Residuals:                     75200   BIC:                        -3.332e+05
Df Model:                             9                                         
Covariance Type:              nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0031  

In [77]:
targets = ["target_log_return_t1", "target_log_return_t2", "target_log_return_t3", "target_log_return_t5"]
price_return_multiday_df = normalize_targets_per_ticker(price_return_multiday_df, targets)
price_return_multiday_df

,Date,Ticker,log_return,close_price,rsi_14,macd_line,macd_signal,macd_hist,volatility_20,ma_50_dist,target_log_return_t1,target_log_return_t2,target_log_return_t3,target_log_return_t5,norm_target_log_return_t1,norm_target_log_return_t2,norm_target_log_return_t3,norm_target_log_return_t5
0,2016-04-01,AAPL,0.009133,24.684103,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010221,-0.001638,0.008780,-0.012166,0.510476,-0.138634,0.196168,-0.431312
1,2016-04-04,AAPL,0.010221,24.910585,0.000000,0.018067,0.003613,0.014454,0.000000,0.000000,-0.011859,-0.001441,-0.023492,-0.019079,-0.701640,-0.130791,-0.856361,-0.609063
2,2016-04-05,AAPL,-0.011859,25.166506,0.000000,0.052431,0.013377,0.039054,0.000000,0.000000,0.010418,-0.011633,-0.010528,0.005721,0.521279,-0.537002,-0.433550,0.028570
3,2016-04-06,AAPL,0.010418,24.869822,0.000000,0.055090,0.021720,0.033371,0.000000,0.000000,-0.022051,-0.020946,-0.017638,0.009686,-1.261150,-0.908167,-0.665442,0.130527
4,2016-04-07,AAPL,-0.022051,25.130268,0.000000,0.077322,0.032840,0.044482,0.000000,0.000000,0.001105,0.004413,0.017354,0.032273,0.010039,0.102517,0.475791,0.711254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75379,2026-03-16,XOM,0.007085,156.119995,60.823913,3.157060,3.526116,-0.369056,0.015956,0.109174,0.009999,0.002287,0.005898,0.024502,0.544064,0.056937,0.151098,0.575392
75380,2026-03-17,XOM,0.009999,157.229996,66.352078,3.374751,3.495843,-0.121092,0.015431,0.111110,-0.007712,-0.004101,0.005401,0.040537,-0.463829,-0.202787,0.134647,0.989976
75381,2026-03-18,XOM,-0.007712,158.809998,68.932039,3.632889,3.523252,0.109637,0.014347,0.116441,0.003611,0.013112,0.022215,0.035347,0.180511,0.497058,0.691361,0.855791
75382,2026-03-19,XOM,0.003611,157.589996,67.107758,3.696411,3.557884,0.138527,0.014350,0.102739,0.009502,0.018604,0.044639,0.044941,0.515789,0.720336,1.433813,1.103825


In [78]:
targets = ["norm_target_log_return_t1", "norm_target_log_return_t2", "norm_target_log_return_t3", "norm_target_log_return_t5"]

for target in targets:
    run_individual_hypothesis_tests(price_return_multiday_df, target)

Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.012133 | 8.7637e-04 | YES
macd_hist       |    -0.002239 | 5.3921e-01 | NO
volatility_20   |     0.011471 | 1.6555e-03 | YES
ma_50_dist      |    -0.011774 | 1.2419e-03 | YES
Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.017397 | 1.8299e-06 | YES
macd_hist       |    -0.003633 | 3.1915e-01 | NO
volatility_20   |     0.018462 | 4.1159e-07 | YES
ma_50_dist      |    -0.021548 | 3.4238e-09 | YES
Indicator       | Correlation  | P-Value    | Significant?
------------------------------------------------------------
rsi_14          |    -0.021252 | 5.5790e-09 | YES
macd_hist       |    -0.002460 | 4.9987e-01 | NO
volatility_20   |     0.024005 | 4.5737e-11 | YES
ma_50_dist      |    -0.026667 | 2.5842e-13 | YES
Indicator       | Correlation  | P-Value   

In [79]:
targets = ["norm_target_log_return_t1", "norm_target_log_return_t2", "norm_target_log_return_t3", "norm_target_log_return_t5"]
for target in targets:
    multivariate_model = run_multivariate_test(price_return_multiday_df, target)
    print(multivariate_model.summary())

                                OLS Regression Results                               
Dep. Variable:     norm_target_log_return_t1   R-squared:                       0.000
Model:                                   OLS   Adj. R-squared:                  0.000
Method:                        Least Squares   F-statistic:                     3.341
Date:                       Tue, 07 Apr 2026   Prob (F-statistic):           0.000428
Time:                               16:20:48   Log-Likelihood:            -1.0669e+05
No. Observations:                      75210   AIC:                         2.134e+05
Df Residuals:                          75200   BIC:                         2.135e+05
Df Model:                                  9                                         
Covariance Type:                   nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------